In [2]:
from langgraph.graph import StateGraph,START,END
from typing import TypedDict,List

In [ ]:
class ApplicantState(TypedDict):
    name: str
    age: int
    year_of_experience:int
    score_of_experiencee:int
    skill_score: int
    skills:list[str]
    feedback:str
    desired_role:str
    fit:str

In [4]:
def validate_input(state:ApplicantState) -> ApplicantState:
    if not state.get("name") or  not state.get('age') or not state.get("year_of_experience")<0:
        return ValueError("invlid iinput")
    return state

In [7]:
def score_of_experience(state: ApplicantState)->ApplicantState:
    exp=state['year_of_experience']
    if exp >=5:
        state['score_of_experiencee']=5
    elif exp >=3:
        state['score_of_experiencee']=3

    else:
        state['score_of_experiencee']=2

    return state         


In [8]:
REQUIRED_SKILLS = {
    "Data Scientist": ["Python", "Pandas", "Machine Learning", "SQL"],
    "Frontend Developer": ["HTML", "CSS", "JavaScript", "React"],
    "Backend Developer": ["Python", "Django", "APIs", "Databases"]
}

In [ ]:
def score_skill(state: ApplicantState)-> ApplicantState:
    role=state['desired_role']
    applicant_skill=set(skill.lower() for skill in state['skills'])
    req_skill=set(skill.lower() for skill in REQUIRED_SKILLS.get(role,[]))
    matched = len(applicant_skill.intersection(req_skill))
    state["skill_score"] = matched
    return state

    
    

In [10]:
def ev_fit(state: ApplicantState)->ApplicantState:
    exp=state['score_of_experiencee']
    skill=state['skill_score']

    if exp>=5 and skill >=3:
        state['fit']='excelent'
    elif exp>=3 and skill >=2:
        state['fit']='avg'

    else:
        state['fit']='poor'   
    return state     


In [11]:
def generate_feedback(state: ApplicantState) -> ApplicantState:
    feedback = f"{state['name']}, you applied for a {state['desired_role']} role. "
    if state["fit"] == "Excellent":
        feedback += "✅ Great fit! You're highly qualified."
    elif state["fit"] == "Good":
        feedback += "👍 You're a decent fit. Consider improving a few areas."
    else:
        feedback += "⚠️ You're currently not a great match. Consider gaining more experience or skills."
    state["feedback"] = feedback
    return state

In [32]:
graph = StateGraph(ApplicantState)

graph.add_node("validate_input", validate_input)
graph.add_node("score_experience", score_of_experience)
graph.add_node("score_skills", score_skill)
graph.add_node("evaluate_fit", ev_fit)
graph.add_node("generate_feedback", generate_feedback)

In [33]:
graph.add_edge(START, "validate_input")
graph.add_edge("validate_input", "score_of_experience")
graph.add_edge("score_of_experience", "score_skill")
graph.add_edge("score_skill","ev_fit")
graph.add_edge("ev_fit", "generate_feedback")
graph.add_edge("generate_feedback", END)

job_workflow = graph.compile()


ValueError: Found edge starting at unknown node 'score_skill'